# ============================================================
# VISUAL MAP OF OUR INCEPTION-C BLOCK
# Inspired by InceptionV3 Mixed 9 / Mixed 10
#
# The purpose is to SEE the same structure that we coded:
#
#                    INPUT
#                      │
#       ┌──────────────┼──────────────┬──────────────┐
#       │              │              │              │
#       ▼              ▼              ▼              ▼
#    Branch 1       Branch 2       Branch 3       Branch 4
#       │              │              │              │
#       └──────────────┴──────┬───────┴──────────────┘
#                             ▼
#                         CONCATENATE
#                             │
#                           OUTPUT
# ============================================================

                              INPUT
                                │
             ┌──────────────────┼──────────────────┬─────────────────┐
             │                  │                  │                 │
             ▼                  ▼                  ▼                 ▼
         BRANCH 1           BRANCH 2           BRANCH 3         BRANCH 4
            1×1                1×1                1×1             AvgPool
                               │                   │                 │
                         ┌─────┴─────┐             3×3              1×1
                         │           │              │
                        1×3         3×1        ┌────┴────┐
                         │           │         │         │
                         └─────┬─────┘        1×3       3×1
                               │                │         │
                               │                └────┬────┘
             │                 │                     │              │
             └─────────────────┴──────────┬──────────┴──────────────┘
                                          ▼
                                      CONCATENATE
                                          │
                                       OUTPUT

In [1]:
import os

# Keras must know which backend to use BEFORE importing keras.
os.environ["KERAS_BACKEND"] = "torch"

import keras

# Functional API model.
from keras import Model

# Every layer that we are going to use below.
from keras.layers import (
    Input,
    Rescaling,

    Conv2D,
    BatchNormalization,
    Activation,

    MaxPooling2D,
    AveragePooling2D,

    Concatenate,

    GlobalAveragePooling2D,
    Dropout,
    Dense,
)

In [2]:
def conv_bn_relu(
    x,
    filters,
    kernel_size,
    name
):
    """
    One reusable Inception-style convolution block.

    x
        = input tensor

    filters
        = number of feature maps we want to create

    kernel_size
        = size of convolution filter:
          (1,1), (3,3), (1,3), etc.

    name
        = readable name so model.summary()
          remains understandable.
    """

    # -----------------------------------
    # 1. CONVOLUTION
    # -----------------------------------
    x = Conv2D(
        filters=filters,
        kernel_size=kernel_size,

        # Keep width and height unchanged.
        padding="same",

        # BatchNorm supplies its own offset,
        # therefore we do not need Conv2D bias.
        use_bias=False,

        name=f"{name}_conv",
    )(x)


    # -----------------------------------
    # 2. BATCH NORMALISATION
    # -----------------------------------
    x = BatchNormalization(
        name=f"{name}_bn"
    )(x)


    # -----------------------------------
    # 3. ACTIVATION
    # -----------------------------------
    x = Activation(
        "relu",
        name=f"{name}_relu"
    )(x)

    return x

tensor
  │
  ▼
Conv2D
learn filters
  │
  ▼
BatchNorm
stabilise values
  │
  ▼
ReLU
non-linearity
  │
  ▼
new tensor

In [3]:
def inception_c_block(x, name):
    """
    Compact CIFAR-10 version of the structure used
    by InceptionV3 Mixed 9 / Mixed 10.

                    INPUT
                      │
       ┌──────────────┼──────────────┬──────────────┐
       │              │              │              │
       ▼              ▼              ▼              ▼
    Branch 1       Branch 2       Branch 3       Branch 4
       │              │              │              │
       └──────────────┴──────┬───────┴──────────────┘
                             ▼
                         CONCATENATE
                             │
                           OUTPUT
    """


    # ======================================================
    # BRANCH 1
    #
    # INPUT
    #   ↓
    # 1×1 Conv
    #
    # Original Inception: 320 filters
    # Compact CIFAR:       40 filters
    # ======================================================

    branch1 = conv_bn_relu(
        x,
        filters=40,
        kernel_size=(1, 1),
        name=f"{name}_branch1",
    )


    # ======================================================
    # BRANCH 2
    #
    # INPUT
    #   ↓
    # 1×1
    #   ↓
    #   SPLIT
    #  /     \
    # 1×3    3×1
    #  \     /
    # CONCAT
    #
    # Output channels:
    # 48 + 48 = 96
    # ======================================================

    branch2_base = conv_bn_relu(
        x,
        filters=48,
        kernel_size=(1, 1),
        name=f"{name}_branch2_base",
    )


    # Horizontal-style convolution.
    branch2_1x3 = conv_bn_relu(
        branch2_base,
        filters=48,
        kernel_size=(1, 3),
        name=f"{name}_branch2_1x3",
    )


    # Vertical-style convolution.
    branch2_3x1 = conv_bn_relu(
        branch2_base,
        filters=48,
        kernel_size=(3, 1),
        name=f"{name}_branch2_3x1",
    )


    # First INTERNAL concatenation.
    branch2 = Concatenate(
        axis=-1,
        name=f"{name}_branch2_concat",
    )([
        branch2_1x3,
        branch2_3x1,
    ])


    # ======================================================
    # BRANCH 3
    #
    # INPUT
    #   ↓
    # 1×1
    #   ↓
    # 3×3
    #   ↓
    #   SPLIT
    #  /     \
    # 1×3    3×1
    #  \     /
    # CONCAT
    #
    # This is the deeper branch.
    #
    # Output:
    # 48 + 48 = 96 channels
    # ======================================================

    branch3 = conv_bn_relu(
        x,
        filters=56,
        kernel_size=(1, 1),
        name=f"{name}_branch3_reduce",
    )


    branch3 = conv_bn_relu(
        branch3,
        filters=48,
        kernel_size=(3, 3),
        name=f"{name}_branch3_3x3",
    )


    branch3_1x3 = conv_bn_relu(
        branch3,
        filters=48,
        kernel_size=(1, 3),
        name=f"{name}_branch3_1x3",
    )


    branch3_3x1 = conv_bn_relu(
        branch3,
        filters=48,
        kernel_size=(3, 1),
        name=f"{name}_branch3_3x1",
    )


    # Second INTERNAL concatenation.
    branch3 = Concatenate(
        axis=-1,
        name=f"{name}_branch3_concat",
    )([
        branch3_1x3,
        branch3_3x1,
    ])


    # ======================================================
    # BRANCH 4
    #
    # INPUT
    #   ↓
    # Average Pooling
    #   ↓
    # 1×1 Conv
    #
    # Original Inception: 192 filters
    # Compact CIFAR:       24 filters
    # ======================================================

    branch4 = AveragePooling2D(
        pool_size=(3, 3),

        # stride=1 means:
        # do NOT reduce spatial dimensions here.
        strides=(1, 1),

        padding="same",

        name=f"{name}_branch4_avgpool",
    )(x)


    branch4 = conv_bn_relu(
        branch4,
        filters=24,
        kernel_size=(1, 1),
        name=f"{name}_branch4_1x1",
    )


    # ======================================================
    # FINAL BLOCK CONCATENATION
    #
    # We now concatenate the FOUR TOP-LEVEL branches:
    #
    # Branch 1 = 40
    # Branch 2 = 96
    # Branch 3 = 96
    # Branch 4 = 24
    #
    # TOTAL = 256 channels
    # ======================================================

    output = Concatenate(
        axis=-1,
        name=f"{name}_output_concat",
    )([
        branch1,
        branch2,
        branch3,
        branch4,
    ])


    return output

In [4]:
cifar10_inception_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="cifar10_inception"
)

cnn.summary()

plot_model(
    cnn,
    show_shapes=True,
    show_layer_names=True,
    expand_nested=True,
    dpi=90,
)

NameError: name 'inputs' is not defined

In [ ]:
from keras.utils import plot_model

plot_model(
    cifar10_inception_model,
    show_shapes=True,
    show_layer_names=True,
    expand_nested=True,
    dpi=90,
)

In [ ]:
# Artificial input having the dimensions
# we expect before our first Inception block.
test_input = Input(
    shape=(8, 8, 128),
    name="test_input"
)

test_output = inception_c_block(
    test_input,
    name="test_inception"
)

test_model = Model(
    inputs=test_input,
    outputs=test_output,
)

test_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ test_input          │ (None, 8, 8, 128) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 56)  │      7,168 │ test_input[0][0]  │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 56)  │        224 │ test_inception_b… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 56)  │          0 │ test_inception_b… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 48)  │      6,144 │ test_input[0][0]  │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 48)  │     24,192 │ test_inception_b… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 48)  │        192 │ test_inception_b… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 48)  │        192 │ test_inception_b… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 48)  │          0 │ test_inception_b… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 48)  │          0 │ test_inception_b… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 48)  │      6,912 │ test_inception_b… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 48)  │      6,912 │ test_inception_b… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 48)  │      6,912 │ test_inception_b… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 48)  │      6,912 │ test_inception_b… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 128) │          0 │ test_input[0][0]  │
│ (AveragePooling2D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 40)  │      5,120 │ test_input[0][0]  │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ test_inception_bra… │ (None, 8, 8, 48)  │        192 │ test_inception_b

 Total params: 74,976 (292.88 KB)

 Trainable params: 74,160 (289.69 KB)

 Non-trainable params: 816 (3.19 KB)